> ### ⚠️ Select the **`Python 3 (croprow)`** kernel first
> This notebook runs in the isolated croprow env (Python **3.11**, OpenCV). If the first cell throws `ModuleNotFoundError: No module named 'cv2'`, the wrong interpreter is selected.
>
> **VSCode:** click **Select Kernel** (top-right) → **Jupyter Kernel** → **`Python 3 (croprow)`**, or **Python Environments... → Enter interpreter path...** and paste `croprow\.venv\Scripts\python.exe`.
>
> Do **not** use the repo-root `.venv` — that is the potato backend (Python 3.13, no cv2 by design). `croprow_disease` shares the `croprow` env; it needs no venv of its own.

# 01 — Dataset prep (LettuceMOTS → YOLO detection, **two classes**)

**Goal:** verify the label format, split train/val **by sequence folder**, derive an axis-aligned box *and* a health class for every annotated plant, and emit the Ultralytics data yaml (`nc=2`, `names=['healthy','unhealthy']`).

Two derivations happen here, both deterministic and both from real data:

| what | from | how |
| --- | --- | --- |
| **box** | the human-drawn LettuceMOTS polygon | min/max of its normalized vertices — identical to what `croprow/` does |
| **class** | the real pixels that polygon encloses | the colour rule in [`health.py`](../health.py): mostly vigorous green → `healthy`, brown/yellowed/off → `unhealthy` |

The class half is an **auto-label, not verified disease ground truth.** Nothing is synthetic — every pixel is a real captured frame and every polygon a real annotation — but a colour heuristic decided healthy vs unhealthy, so every metric downstream is agreement with that rule, not clinical accuracy. Say so wherever you quote a number.

Instances too blurred, too dark, or too small to judge on colour are **not** labelled unhealthy — they default to `healthy` and are counted as low-confidence. That gate exists because without it the rule labels motion blur as disease; see the measurement table in `health.py`.

Split is **by sequence folder**: frames within a sequence are consecutive video frames, so a whole sequence goes entirely to train or entirely to val. Random per-frame splitting would leak near-identical adjacent frames across the split. Same seed and fraction as `croprow/`, so the two modules stay comparable.

> This notebook reads **every frame** (the colour rule needs pixels), so conversion takes minutes, not seconds.

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

# === CONFIG: the ONLY place to set the dataset location ===================
LETTUCE_ROOT_DEFAULT = r"D:\croprow_dataset\LettuceMOTS"
LETTUCE_ROOT = os.environ.get("LETTUCE_ROOT", LETTUCE_ROOT_DEFAULT)

# Colour-rule thresholds. Defaults are measured on LettuceMOTS (see health.py);
# retune here for a different crop or camera -- never edit the module.
PARAMS = HealthParams()

VAL_FRAC = 0.25
SEED = 42
# ==========================================================================

root = U.resolve_lettuce_root(LETTUCE_ROOT)
print("LETTUCE_ROOT :", root)
print("DATA_DIR     :", DATA_DIR)
print("labels ->    :", U.box_labels_dir(root))
print()
for k, v in PARAMS.as_dict().items():
    print(f"  {k:22s}: {v}")

## 1. Verify label format

A detection box line is exactly 5 tokens (`class cx cy w h`); a segmentation polygon line is an odd count > 5. The class ids present tell us which case we are in.

In [ ]:
fmt = U.inspect_label_format(root)
for k, v in fmt.items():
    print(f"  {k:20s}: {v}")

if fmt["is_two_class_boxes"]:
    TRUST_LABELS = True
    print("\nLabels are REAL 2-class boxes -> passed through unchanged, "
          "the colour rule is not used.")
elif fmt["is_polygons"]:
    TRUST_LABELS = False
    print("\nLabels are single-class segmentation polygons -> deriving boxes "
          "(geometry) and health classes (colour rule).")
elif fmt["is_boxes"]:
    TRUST_LABELS = False
    print("\nLabels are single-class boxes -> deriving health classes from the "
          "box interior. Looser than the polygon path (a box includes soil).")
else:
    raise RuntimeError(
        f"Unexpected label format (box={fmt['box_lines']}, "
        f"poly={fmt['polygon_lines']}, other={fmt['other_lines']}). "
        "Assumption failed -- stop and inspect the dataset.")

## 2. Sequences and split (by folder)

In [ ]:
seqs = U.labeled_sequences(root)
print("labeled train sequences:", seqs)
print("test sequences (no labels, not used for training):", U.test_sequences(root))

train_seqs, val_seqs = U.split_sequences(seqs, val_frac=VAL_FRAC, seed=SEED)
print("\nTRAIN sequences:", train_seqs)
print("VAL   sequences:", val_seqs)
assert not (set(train_seqs) & set(val_seqs)), "sequence leaked across split!"

## 3. Preview the health-score distribution *before* converting

Scores a random sample so `green_frac_threshold` can be sanity-checked against what the crop actually looks like, before spending minutes writing labels. `borderline` is the share of instances sitting within ±0.05 of the cut — if that is large, the threshold is slicing through the middle of one population and the split is arbitrary rather than meaningful.

In [ ]:
rep = U.label_report(root, seqs, params=PARAMS, sample_frames=120, seed=1)
print(f"sampled {rep['frames_scanned']} frames -> {rep['instances']} instances\n")
print(f"  healthy       : {rep['healthy']:6d}  ({rep['healthy_pct']:5.1f}%)")
print(f"  unhealthy     : {rep['unhealthy']:6d}  ({rep['unhealthy_pct']:5.1f}%)")
print(f"  low-confidence: {rep['low_confidence']:6d}  "
      f"({100*rep['low_confidence']/max(rep['instances'],1):5.1f}%)  "
      f"<- too blurred/dark/small to judge, defaulted to healthy")
print(f"\n  score median  : {rep['score_median']:.3f}   "
      f"p05 {rep['score_p05']:.3f}   p95 {rep['score_p95']:.3f}")
print(f"  borderline    : {rep['borderline']} "
      f"({rep['borderline_pct']:.1f}% within +/-{rep['borderline_margin']} of "
      f"the {PARAMS.green_frac_threshold} cut)")

print("\nhealth-score histogram:")
edges, hist = rep["bin_edges"], rep["hist"]
peak = max(hist) or 1
for i, c in enumerate(hist):
    print(f"  {edges[i]:.2f}-{edges[i+1]:.2f} {c:6d} {'#' * int(58 * c / peak)}")

## 4. Convert → 2-class box labels

Written to `<LETTUCE_ROOT>/train/labels_health/<seq>/<frame>.txt`.

Note the `labels_health` name: `croprow/` writes *single-class* labels into `train/labels/` for these same frames, and Ultralytics finds labels by swapping `images` → `labels` in the image path. Sharing the directory would mean whichever module ran last silently decides what the other one trains on.

In [ ]:
per_seq = U.convert_all(root, seqs, params=PARAMS, trust_label_class=TRUST_LABELS)

totals = {"healthy": 0, "unhealthy": 0, "frames": 0, "low_confidence": 0}
for c in per_seq.values():
    for k in totals:
        totals[k] += c[k]
print(f"\n  TOTAL: {totals['frames']} frames | healthy {totals['healthy']} | "
      f"unhealthy {totals['unhealthy']} | low-conf {totals['low_confidence']}")

## 5. Class-balance gate — is this actually trainable?

A detector needs a real population of **both** classes. This cell refuses to pretend otherwise.

In [ ]:
verdict = U.check_class_balance(totals)
print(verdict["message"])
if not verdict["ok"]:
    print("\n" + "=" * 72)
    print("The yaml and label files below are still written -- they are correct,")
    print("and they are exactly what notebook 04 fine-tunes FROM. But do not")
    print("train a two-class model on this split alone and quote its mAP.")
    print("=" * 72)

## 6. Write train.txt / val.txt and the data yaml

In [ ]:
train_imgs = U.image_paths_for_seqs(root, train_seqs)
val_imgs   = U.image_paths_for_seqs(root, val_seqs)
n_tr = U.write_list_file(DATA_DIR / "train.txt", train_imgs)
n_va = U.write_list_file(DATA_DIR / "val.txt", val_imgs)
print(f"train.txt: {n_tr} images\nval.txt  : {n_va} images")

yaml_path = U.make_data_yaml(DATA_DIR / "health.yaml",
                             DATA_DIR / "train.txt", DATA_DIR / "val.txt")
print("\nwrote", yaml_path, "\n")
print(yaml_path.read_text())

## 7. Per-split summary (images + per-class instances)

In [ ]:
def summarize(name, split_seqs):
    imgs = U.image_paths_for_seqs(root, split_seqs)
    c = U.count_instances_for_seqs(root, split_seqs)
    print(f"{name:5s} | seqs={len(split_seqs):2d} | images={len(imgs):5d} | "
          f"healthy={c['healthy']:6d} | unhealthy={c['unhealthy']:6d}")
    return len(imgs), c

print("split | #seqs | #images | healthy | unhealthy")
print("-" * 62)
tr_n, tr_c = summarize("train", train_seqs)
va_n, va_c = summarize("val", val_seqs)
print("-" * 62)
print(f"TOTAL | seqs={len(seqs):2d} | images={tr_n + va_n:5d} | "
      f"healthy={tr_c['healthy'] + va_c['healthy']:6d} | "
      f"unhealthy={tr_c['unhealthy'] + va_c['unhealthy']:6d}")